# Convert SQLite outputs to parquet files with CytoTable

## Import libraries

In [1]:
import logging
import pathlib
import pprint

import pandas as pd

# cytotable will merge objects from SQLite file into single cells and save as parquet file
from cytotable import convert, presets

# Set the logging level to a higher level to avoid outputting unnecessary errors from config file in convert function
logging.getLogger().setLevel(logging.ERROR)


## Set paths and variables

In [2]:
# base directory where the validation plate's CellProfiler outputs are located
base_dir = pathlib.Path("../1.cellprofiler_processing/cp_output/").resolve(strict=True)

# find all plate folders that contain a CellProfiler SQLite output
plate_names = [
    p.name for p in base_dir.glob("CARD*") if p.is_dir() and list(p.glob("*.sqlite"))
]

print(f"There are a total of {len(plate_names)} plate(s) to convert:")
for plate in plate_names:
    print(plate)


There are a total of 5 plate(s) to convert:
CARD-CelIns-CX7_261101160003
CARD-CelIns-CX7_261030160001
CARD-CelIns-CX7_261102080001
CARD-CelIns-CX7_261031160001
CARD-CelIns-CX7_261102150001


In [3]:
# preset configurations based on typical CellProfiler outputs
preset = "cellprofiler_sqlite_pycytominer"

# update preset to include site metadata and cell counts
joins = presets.config[preset]["CONFIG_JOINS"].replace(
    "Image_Metadata_Well,",
    "Image_Metadata_Well, Image_Metadata_Site, Image_Count_Cells,",
)

# Add the PathName columns separately
joins = joins.replace(
    "COLUMNS('Image_FileName_.*'),",
    "COLUMNS('Image_FileName_.*'),\n COLUMNS('Image_PathName_.*'),",
)

# type of file output from cytotable
dest_datatype = "parquet"

# directory for processed data
output_base = pathlib.Path("data")
converted_dir = output_base / "converted_profiles"
converted_dir.mkdir(parents=True, exist_ok=True)

# create plate info dictionary with paths needed for conversion
plate_info_dictionary = {
    name: {
        "path_to_sqlite": pathlib.Path(
            next(iter((base_dir / name).glob("*.sqlite")))
        ).resolve(strict=True),
        "path_to_output": converted_dir / f"{name}_converted.parquet",
    }
    for name in plate_names
}

# view the dictionary to assess that all info is added correctly
pprint.pprint(plate_info_dictionary, indent=4)

converted_profiles_exist = bool(plate_info_dictionary) and all(
    info["path_to_output"].is_file() for info in plate_info_dictionary.values()
)

if converted_profiles_exist:
    print(
        "Converted profiles already exist for all plates. Conversion cells will be skipped."
    )


{   'CARD-CelIns-CX7_261030160001': {   'path_to_output': PosixPath('data/converted_profiles/CARD-CelIns-CX7_261030160001_converted.parquet'),
                                        'path_to_sqlite': PosixPath('/home/jenna/targeted_fibrosis_drug_screen/5.validation_plate_profiling/1.cellprofiler_processing/cp_output/CARD-CelIns-CX7_261030160001/cfret_features.sqlite')},
    'CARD-CelIns-CX7_261031160001': {   'path_to_output': PosixPath('data/converted_profiles/CARD-CelIns-CX7_261031160001_converted.parquet'),
                                        'path_to_sqlite': PosixPath('/home/jenna/targeted_fibrosis_drug_screen/5.validation_plate_profiling/1.cellprofiler_processing/cp_output/CARD-CelIns-CX7_261031160001/cfret_features.sqlite')},
    'CARD-CelIns-CX7_261101160003': {   'path_to_output': PosixPath('data/converted_profiles/CARD-CelIns-CX7_261101160003_converted.parquet'),
                                        'path_to_sqlite': PosixPath('/home/jenna/targeted_fibrosis_drug_scree

## Convert SQLite to parquet files

In [4]:
if converted_profiles_exist:
    print(
        "Skipping SQLite to parquet conversion because converted profiles already exist."
    )
else:
    for plate_name, info in plate_info_dictionary.items():
        print("Starting conversion with cytotable for plate:", plate_name)
        # Merge single cells and output as parquet file
        convert(
            source_path=str(info["path_to_sqlite"]),
            dest_path=str(info["path_to_output"]),
            dest_datatype=dest_datatype,
            preset=preset,
            joins=joins,
            chunk_size=5000,
        )

    print("All plates have been converted with cytotable!")


Starting conversion with cytotable for plate: CARD-CelIns-CX7_261101160003
Starting conversion with cytotable for plate: CARD-CelIns-CX7_261030160001
Starting conversion with cytotable for plate: CARD-CelIns-CX7_261102080001
Starting conversion with cytotable for plate: CARD-CelIns-CX7_261031160001
Starting conversion with cytotable for plate: CARD-CelIns-CX7_261102150001
All plates have been converted with cytotable!


# Load in converted profiles to update

We will rename some of the columns (e.g., location centroids and cell count per FOV, image FileName and PathName columns) to include Metadata prefix.

In [5]:
if converted_profiles_exist:
    print("Skipping metadata updates because converted profiles already exist.")
else:
    # List of columns to update with the "Metadata_" prefix
    metadata_columns_to_update = [
        "Nuclei_Location_Center_X",
        "Nuclei_Location_Center_Y",
        "Cells_Location_Center_X",
        "Cells_Location_Center_Y",
        "Image_Count_Cells",
    ]

    for plate_name, info in plate_info_dictionary.items():
        file_path = info["path_to_output"]
        if not file_path.is_file():
            print(f"Warning: file not found for plate {plate_name}")
            continue

        # Load the DataFrame from the Parquet file
        df = pd.read_parquet(file_path)

        # Ensure Metadata_Plate contains only one unique value (occurs due to failure acquiring plate during run)
        if "Metadata_Plate" in df.columns and df["Metadata_Plate"].nunique() != 1:
            df["Metadata_Plate"] = plate_name

        # Drop rows where "Metadata_ImageNumber" is NaN
        df = df.dropna(subset=["Metadata_ImageNumber"])

        # Add the image FileName and PathName columns (one per channel) to the update list
        plate_metadata_columns = metadata_columns_to_update + [
            col
            for col in df.columns
            if col.startswith(("Image_FileName_", "Image_PathName_"))
        ]

        # Rearrange columns and add "Metadata_" prefix
        df = df[
            plate_metadata_columns
            + [col for col in df.columns if col not in plate_metadata_columns]
        ].rename(
            columns=lambda col: (
                "Metadata_" + col if col in plate_metadata_columns else col
            )
        )

        # Save the processed DataFrame back to the same path
        df.to_parquet(file_path, index=False)
        print(f"Processed metadata columns for plate: {plate_name}")

    print("All converted profiles have been updated with Metadata columns!")


Processed metadata columns for plate: CARD-CelIns-CX7_261101160003
Processed metadata columns for plate: CARD-CelIns-CX7_261030160001
Processed metadata columns for plate: CARD-CelIns-CX7_261102080001
Processed metadata columns for plate: CARD-CelIns-CX7_261031160001
Processed metadata columns for plate: CARD-CelIns-CX7_261102150001
All converted profiles have been updated with Metadata columns!


## Check output to confirm process worked

To confirm the number of single cells is correct, please use any database browser software to see if the number of rows in the "Per_Cells" compartment matches the number of rows in the data frame.

In [6]:
# pick a plate to inspect (there is currently only one validation plate)
first_plate = next(iter(plate_info_dictionary.keys()))
converted_path = plate_info_dictionary[first_plate]["path_to_output"]

# Load the selected converted parquet file
converted_df = pd.read_parquet(converted_path)

print(f"Loaded file: {converted_path}")
print(converted_df.shape)
converted_df.head()


Loaded file: data/converted_profiles/CARD-CelIns-CX7_261101160003_converted.parquet
(16013, 2486)


,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_Image_FileName_IllumActin,Metadata_Image_FileName_IllumDNA,Metadata_Image_FileName_IllumER,Metadata_Image_FileName_IllumGolgi,Metadata_Image_FileName_IllumMito,...,Nuclei_Texture_Variance_ER_3_02_256,Nuclei_Texture_Variance_ER_3_03_256,Nuclei_Texture_Variance_Golgi_3_00_256,Nuclei_Texture_Variance_Golgi_3_01_256,Nuclei_Texture_Variance_Golgi_3_02_256,Nuclei_Texture_Variance_Golgi_3_03_256,Nuclei_Texture_Variance_Mito_3_00_256,Nuclei_Texture_Variance_Mito_3_01_256,Nuclei_Texture_Variance_Mito_3_02_256,Nuclei_Texture_Variance_Mito_3_03_256
0,519.203663,42.410256,511.942406,77.532363,15,CARD-CelIns-CX7_261101160003_IllumActin.npy,CARD-CelIns-CX7_261101160003_IllumDNA.npy,CARD-CelIns-CX7_261101160003_IllumER.npy,CARD-CelIns-CX7_261101160003_IllumGolgi.npy,CARD-CelIns-CX7_261101160003_IllumMito.npy,...,8.667805,7.352031,7.210999,6.474589,6.788036,7.047577,0.626135,0.554609,0.580732,0.646687
1,673.118998,159.069937,678.977593,154.363376,13,CARD-CelIns-CX7_261101160003_IllumActin.npy,CARD-CelIns-CX7_261101160003_IllumDNA.npy,CARD-CelIns-CX7_261101160003_IllumER.npy,CARD-CelIns-CX7_261101160003_IllumGolgi.npy,CARD-CelIns-CX7_261101160003_IllumMito.npy,...,4.541035,3.517272,5.591936,5.140991,4.981243,5.243994,2.161239,2.298288,2.223068,2.029088
2,751.711523,86.921031,716.935459,105.332678,12,CARD-CelIns-CX7_261101160003_IllumActin.npy,CARD-CelIns-CX7_261101160003_IllumDNA.npy,CARD-CelIns-CX7_261101160003_IllumER.npy,CARD-CelIns-CX7_261101160003_IllumGolgi.npy,CARD-CelIns-CX7_261101160003_IllumMito.npy,...,1.509835,1.360538,9.937514,9.074070,9.005738,8.843156,1.803021,1.792765,1.891013,1.790089
3,593.890511,106.040146,593.721454,99.173402,16,CARD-CelIns-CX7_261101160003_IllumActin.npy,CARD-CelIns-CX7_261101160003_IllumDNA.npy,CARD-CelIns-CX7_261101160003_IllumER.npy,CARD-CelIns-CX7_261101160003_IllumGolgi.npy,CARD-CelIns-CX7_261101160003_IllumMito.npy,...,1.503122,1.486497,3.397710,3.464839,3.462980,3.582072,1.618157,1.502250,1.542174,1.690843
4,325.693560,92.298298,280.188305,132.451258,13,CARD-CelIns-CX7_261101160003_IllumActin.npy,CARD-CelIns-CX7_261101160003_IllumDNA.npy,CARD-CelIns-CX7_261101160003_IllumER.npy,CARD-CelIns-CX7_261101160003_IllumGolgi.npy,CARD-CelIns-CX7_261101160003_IllumMito.npy,...,2.468253,2.299414,9.527799,8.567628,8.455459,8.628596,1.136587,1.157684,1.133447,1.124073
